# Steps 1-3: CelebA, Offline CLIP Features, and Zero-Shot Baselines

This notebook implements the first three stages requested by the assignment: data exploration, frozen offline feature extraction, and a vanilla CLIP retrieval baseline. It uses the required Hugging Face model **openai/clip-vit-base-patch32** and evaluates retrieved test indices against the official **celeba_evaluation.json**.

In [ ]:
%pip install -q transformers torchvision pandas matplotlib tqdm

## Configuration

The project root must contain the CelebA folder and the official evaluation JSON. Embeddings are stored under **data/celeba/embeddings/**; benchmark outputs are stored under **artifacts/results/baselines/**.

In [ ]:
from pathlib import Path
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import CelebA
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

MODEL_ID = "openai/clip-vit-base-patch32"
MODEL_SLUG = "openai_clip_vit_b32"
BATCH_SIZE = 128
NUM_WORKERS = 2
TOP_KS = (1, 5, 10)

def find_project_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/drive/MyDrive/deep_learning")]
    for candidate in candidates:
        if (candidate / "celeba_evaluation.json").exists() or (candidate / "data" / "celeba_evaluation.json").exists():
            return candidate
    raise FileNotFoundError("Set PROJECT_ROOT to the folder containing celeba_evaluation.json")

PROJECT_ROOT = find_project_root()
DATASET_ROOT = PROJECT_ROOT / "data" if (PROJECT_ROOT / "data" / "celeba").exists() else PROJECT_ROOT
EVALUATION_PATH = (PROJECT_ROOT / "data" / "celeba_evaluation.json") if (PROJECT_ROOT / "data" / "celeba_evaluation.json").exists() else (PROJECT_ROOT / "celeba_evaluation.json")
EMBEDDING_DIR = PROJECT_ROOT / "data" / "celeba" / "embeddings" / MODEL_SLUG
RESULTS_ROOT = PROJECT_ROOT / "artifacts" / "results" / "baselines"
EMBEDDING_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
print("Project root:", PROJECT_ROOT)
print("Device:", DEVICE)
print("Embedding directory:", EMBEDDING_DIR)
print("Results directory:", RESULTS_ROOT)

# Step 1 - Data Exploration and Preprocessing

The benchmark uses indices of the PyTorch CelebA **test split**, not physical filename numbers. We first inspect the test split, official queries, and valid target lists.

In [ ]:
celeba_test = CelebA(
    root=DATASET_ROOT,
    split="test",
    target_type="attr",
    download=False,
)
with EVALUATION_PATH.open() as handle:
    annotations = json.load(handle)

assert len(celeba_test) == 19_962
print("Test images:", len(celeba_test))
print("Official query entries:", len(annotations))
print("Dataset index 13 maps to filename:", celeba_test.filename[13])

### Official benchmark summary

Each JSON entry maps a signed query to valid source indices and acceptable target indices. The JSON is authoritative, including duplicated query entries.

In [ ]:
query_summary = []
for query_id, item in enumerate(annotations):
    target_counts = [len(targets) for targets in item["ground_truth"].values()]
    query_summary.append({
        "query_id": query_id,
        "query": item["query"],
        "sources": len(target_counts),
        "mean_valid_targets": sum(target_counts) / len(target_counts),
        "min_valid_targets": min(target_counts),
        "max_valid_targets": max(target_counts),
    })
pd.DataFrame(query_summary)

### Query parsing and sanity check

Positive conditions require an attribute to be present; negative conditions require it to be absent. The limited check below confirms that official targets satisfy their query conditions.

In [ ]:
attribute_names = [name for name in celeba_test.attr_names if name]
assert len(attribute_names) == 40
attribute_to_index = {name: index for index, name in enumerate(attribute_names)}

def parse_query(query):
    conditions = []
    for raw_condition in query.split(","):
        token = raw_condition.strip()
        sign = 1 if token[0] == "+" else -1
        conditions.append((sign, token[1:].strip()))
    return conditions

def satisfies_query(attribute_row, conditions):
    for sign, attribute in conditions:
        value = int(attribute_row[attribute_to_index[attribute]])
        expected = 1 if sign > 0 else 0
        if value != expected:
            return False
    return True

checked = 0
for item in annotations:
    conditions = parse_query(item["query"])
    for source_key, valid_targets in item["ground_truth"].items():
        assert valid_targets
        for target_index in valid_targets[:3]:
            _, target_attributes = celeba_test[target_index]
            assert satisfies_query(target_attributes, conditions)
        checked += 1
        if checked >= 100:
            break
    if checked >= 100:
        break
print(f"Validated query constraints for {checked} source cases.")

# Step 2 - Frozen Offline CLIP Feature Extraction

CLIP remains frozen. Image embeddings are computed once, L2-normalized, and stored in test-dataset order. Therefore embedding row **i** always corresponds to **celeba_test[i]**.

In [ ]:
processor = CLIPProcessor.from_pretrained(MODEL_ID)
model = CLIPModel.from_pretrained(MODEL_ID).to(DEVICE).eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
def unwrap_features(output):
    """Support both Transformers 4.x tensor and 5.x model-output APIs."""
    return output.pooler_output if hasattr(output, "pooler_output") else output

print("Projection dimension:", model.projection_dim)

### Test image embedding cache

An existing compatible cache is reused. Otherwise, all 19,962 test images are encoded and saved in float16 to reduce storage. Retrieval later converts them to float32.

In [ ]:
IMAGE_CACHE_PATH = EMBEDDING_DIR / "test_image_embeddings.pt"

class CelebAImageDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, index):
        image, _ = self.dataset[index]
        return image, index

def collate_images(batch):
    images, indices = zip(*batch)
    pixels = processor(images=list(images), return_tensors="pt")["pixel_values"]
    return pixels, torch.tensor(indices)

def load_torch_cache(path):
    try:
        return torch.load(path, map_location="cpu", weights_only=False)
    except TypeError:
        return torch.load(path, map_location="cpu")

if IMAGE_CACHE_PATH.exists():
    image_cache = load_torch_cache(IMAGE_CACHE_PATH)
    assert image_cache["model_id"] == MODEL_ID
    assert len(image_cache["embeddings"]) == len(celeba_test)
else:
    loader = DataLoader(
        CelebAImageDataset(celeba_test),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=DEVICE.type == "cuda",
        collate_fn=collate_images,
    )
    embedding_batches = []
    index_batches = []
    with torch.inference_mode():
        for pixel_values, indices in tqdm(loader, desc="Encoding test images"):
            features = unwrap_features(model.get_image_features(pixel_values=pixel_values.to(DEVICE)))
            embedding_batches.append(F.normalize(features.float(), dim=-1).cpu())
            index_batches.append(indices)

    test_image_embeddings = torch.cat(embedding_batches)
    extracted_indices = torch.cat(index_batches)
    assert torch.equal(extracted_indices, torch.arange(len(celeba_test)))
    image_cache = {
        "model_id": MODEL_ID,
        "split": "test",
        "embeddings": test_image_embeddings.half(),
        "filenames": list(celeba_test.filename),
    }
    torch.save(image_cache, IMAGE_CACHE_PATH)

gallery_embeddings = image_cache["embeddings"].float()
print("Gallery shape:", tuple(gallery_embeddings.shape))
print("Saved at:", IMAGE_CACHE_PATH)

### Text embeddings and contrastive directions

For each attribute we cache a positive prompt, a negative prompt, and the normalized direction **d = normalize(t_positive - t_negative)**. This supports both the required arithmetic baseline and our additional experiments.

In [ ]:
TEXT_CACHE_PATH = EMBEDDING_DIR / "attribute_text_embeddings.pt"

SPECIAL_PROMPTS = {
    "Attractive": ("an attractive face", "an unattractive face"),
    "Bald": ("a bald person", "a person with hair"),
    "Blurry": ("a blurry face photo", "a sharp face photo"),
    "Chubby": ("a chubby face", "a slim face"),
    "Male": ("a male face", "a female face"),
    "Mouth_Slightly_Open": ("a face with an open mouth", "a face with a closed mouth"),
    "No_Beard": ("a clean-shaven face without a beard", "a face with a beard"),
    "Smiling": ("a smiling face", "a face that is not smiling"),
    "Young": ("a young face", "an older face"),
}

def prompts_for(attribute):
    if attribute in SPECIAL_PROMPTS:
        return SPECIAL_PROMPTS[attribute]
    readable = attribute.replace("_", " ").lower()
    return f"a face with {readable}", f"a face without {readable}"

attribute_names = [name for name in celeba_test.attr_names if name]
assert len(attribute_names) == 40
prompt_pairs = [prompts_for(attribute) for attribute in attribute_names]
flat_prompts = [prompt for pair in prompt_pairs for prompt in pair]

if TEXT_CACHE_PATH.exists():
    text_cache = load_torch_cache(TEXT_CACHE_PATH)
    assert text_cache["model_id"] == MODEL_ID
    assert text_cache["attributes"] == attribute_names
else:
    text_inputs = processor(text=flat_prompts, return_tensors="pt", padding=True).to(DEVICE)
    with torch.inference_mode():
        text_features = unwrap_features(model.get_text_features(**text_inputs)).float()
        text_features = F.normalize(text_features, dim=-1).cpu()
    positive = text_features[0::2]
    negative = text_features[1::2]
    directions = F.normalize(positive - negative, dim=-1)
    text_cache = {
        "model_id": MODEL_ID,
        "attributes": attribute_names,
        "prompts": prompt_pairs,
        "positive": positive.half(),
        "negative": negative.half(),
        "directions": directions.half(),
    }
    torch.save(text_cache, TEXT_CACHE_PATH)

positive_text_embeddings = text_cache["positive"].float()
negative_text_embeddings = text_cache["negative"].float()
attribute_directions = text_cache["directions"].float()
print("Text cache shape:", tuple(attribute_directions.shape))
print("Saved at:", TEXT_CACHE_PATH)

# Step 3 - Zero-Shot CLIP Baselines

The assignment requires direct arithmetic: **q = normalize(v_ref + sum(sign * t_attribute))**. To investigate CLIP more thoroughly and prepare for our later fusion method, we also test normalization order and contrastive text directions.

1. **Direct sum (required):** add or subtract positive text embeddings, then normalize once.
2. **Direct sequential:** apply one signed edit at a time, normalizing after each operation.
3. **Contrastive-direction sum:** use normalize(t_with - t_without) for each attribute.
4. **Contrastive-direction sequential:** apply those directions one at a time.

The sequential variants are intentionally order-dependent exploratory baselines.

In [ ]:
METHODS = {
    "direct_sum": "01_direct_sum",
    "direct_sequential": "02_direct_sequential",
    "contrastive_sum": "03_contrastive_sum",
    "contrastive_sequential": "04_contrastive_sequential",
}
text_attribute_to_index = {name: index for index, name in enumerate(attribute_names)}

def compose_queries(source_embeddings, conditions, method):
    query = source_embeddings.float().clone()
    use_directions = method.startswith("contrastive")
    sequential = method.endswith("sequential")
    bank = attribute_directions if use_directions else positive_text_embeddings

    if sequential:
        for sign, attribute in conditions:
            edit = sign * bank[text_attribute_to_index[attribute]].to(query.device)
            query = F.normalize(query + edit, dim=-1)
        return query

    edit = torch.zeros_like(query)
    for sign, attribute in conditions:
        edit = edit + sign * bank[text_attribute_to_index[attribute]].to(query.device)
    return F.normalize(query + edit, dim=-1)

example_source = gallery_embeddings[13:14]
example_conditions = parse_query("+Black_Hair, -Wavy_Hair")
for method in METHODS:
    print(method, compose_queries(example_source, example_conditions, method).shape)

### Official metrics and saved results

Each method writes a separate folder containing per-query metrics, a macro summary, top-10 retrievals, and configuration metadata. Recall@K follows the official hit-rate definition. The source image is excluded.

In [ ]:
def metrics_for_ranking(ranked_indices, valid_targets, k):
    hits = set(ranked_indices[:k]).intersection(valid_targets)
    return int(bool(hits)), len(hits) / k

def run_official_benchmark(method, source_batch_size=256):
    method_dir = RESULTS_ROOT / METHODS[method]
    method_dir.mkdir(parents=True, exist_ok=True)
    gallery = gallery_embeddings.to(DEVICE)
    per_query_rows = []

    with (method_dir / "retrievals.jsonl").open("w") as retrieval_file:
        for query_id, item in enumerate(tqdm(annotations, desc=method)):
            query = item["query"]
            conditions = parse_query(query)
            source_indices = [int(index) for index in item["ground_truth"]]
            totals = {f"Recall@{k}": 0.0 for k in TOP_KS}
            totals.update({f"Precision@{k}": 0.0 for k in TOP_KS})

            for start in range(0, len(source_indices), source_batch_size):
                batch_indices = source_indices[start:start + source_batch_size]
                query_batch = compose_queries(gallery[batch_indices], conditions, method)
                scores = query_batch @ gallery.T
                rows = torch.arange(len(batch_indices), device=DEVICE)
                scores[rows, torch.tensor(batch_indices, device=DEVICE)] = -torch.inf
                rankings = scores.topk(max(TOP_KS), dim=1).indices.cpu().tolist()

                for source_index, ranking in zip(batch_indices, rankings):
                    valid_targets = set(item["ground_truth"][str(source_index)])
                    retrieval_file.write(json.dumps({
                        "query_id": query_id,
                        "query": query,
                        "source_index": source_index,
                        "top10": ranking,
                    }) + "\n")
                    for k in TOP_KS:
                        recall, precision = metrics_for_ranking(ranking, valid_targets, k)
                        totals[f"Recall@{k}"] += recall
                        totals[f"Precision@{k}"] += precision

            count = len(source_indices)
            per_query_rows.append({
                "query_id": query_id,
                "query": query,
                "sources": count,
                **{name: value / count for name, value in totals.items()},
            })

    per_query = pd.DataFrame(per_query_rows)
    metric_columns = [column for column in per_query if "@" in column]
    total_sources = per_query["sources"].sum()
    summary = pd.DataFrame([{
        "method": method,
        "source_query_cases": int(total_sources),
        **{f"macro_{column}": per_query[column].mean() for column in metric_columns},
        **{f"micro_{column}": (per_query[column] * per_query["sources"]).sum() / total_sources for column in metric_columns},
    }])
    per_query.to_csv(method_dir / "per_query_metrics.csv", index=False)
    summary.to_csv(method_dir / "summary.csv", index=False)
    with (method_dir / "config.json").open("w") as handle:
        json.dump({"model_id": MODEL_ID, "method": method, "top_ks": TOP_KS}, handle, indent=2)
    return summary, per_query

### Full benchmark execution

This cell is disabled intentionally. Enable it on the GPU cluster after creating the embedding caches. It evaluates all official source/query cases and saves the four experiments separately.

In [ ]:
RUN_BENCHMARKS = False

if RUN_BENCHMARKS:
    summaries = []
    for method in METHODS:
        summary, _ = run_official_benchmark(method)
        summaries.append(summary)
    all_summaries = pd.concat(summaries, ignore_index=True)
    all_summaries.to_csv(RESULTS_ROOT / "all_methods_summary.csv", index=False)
    display(all_summaries)
else:
    print("Benchmark skipped. Set RUN_BENCHMARKS = True on the cluster.")

## Practical comparison on one official query

The qualitative example applies every composition rule to the same source/query pair. Green indices are accepted by the official JSON; red indices are not.

In [ ]:
def retrieve_one(source_index, query, method, top_k=5):
    gallery = gallery_embeddings.to(DEVICE)
    source = gallery[source_index:source_index + 1]
    composed = compose_queries(source, parse_query(query), method)
    scores = (composed @ gallery.T).squeeze(0)
    scores[source_index] = -torch.inf
    return scores.topk(top_k).indices.cpu().tolist()

def show_example(query_id=9, source_index=None, top_k=5):
    item = annotations[query_id]
    if source_index is None:
        source_index = int(next(iter(item["ground_truth"])))
    valid_targets = set(item["ground_truth"][str(source_index)])
    print(f"Query: {item['query']} | source index: {source_index}")

    for method in METHODS:
        ranking = retrieve_one(source_index, item["query"], method, top_k)
        fig, axes = plt.subplots(1, top_k + 1, figsize=(3 * (top_k + 1), 3))
        axes[0].imshow(celeba_test[source_index][0])
        axes[0].set_title(f"Source\n{source_index}")
        axes[0].axis("off")
        for axis, target_index in zip(axes[1:], ranking):
            axis.imshow(celeba_test[target_index][0])
            color = "green" if target_index in valid_targets else "red"
            axis.set_title(str(target_index), color=color)
            axis.axis("off")
        fig.suptitle(method)
        plt.tight_layout()
        plt.show()

RUN_EXAMPLE = False
if RUN_EXAMPLE:
    show_example(query_id=9, top_k=5)

# Results and Discussion

**Placeholder after cluster execution.** Report Recall@1/5/10 and Precision@1/5/10 per query and as macro averages. Compare the required direct arithmetic baseline with the sequential and contrastive-direction variants, then discuss negation, attribute binding, and edit-order failures.